
Exemple complet d'un modèle KNN (K plus proches voisins) avec scikit-learn


Cas d'usage : détecter si une tumeur du sein est maligne ou bénigne à partir
de 30 mesures prises sur une image de biopsie (jeu de données Breast Cancer
Wisconsin, fourni avec scikit-learn : 569 patientes).

Le script montre, dans l'ordre :
  1. Préparer les données (séparation entraînement / test)
  2. Un premier KNN SANS normalisation (piège classique)
  3. Le même KNN AVEC normalisation
  4. Les métriques d'évaluation (accuracy, précision, rappel, F1, AUC, matrice de confusion)
  5. Mesurer l'erreur : erreur d'entraînement vs erreur de validation selon K
  6. Gérer l'erreur : choisir K et la distance par validation croisée (GridSearchCV)
  7. Gérer l'erreur la plus grave : réduire les faux négatifs avec le seuil
  8. Bonus : KNN en régression et ses métriques (MAE, RMSE, R²)



In [1]:

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
)

SEED = 42


def titre(texte):
    print("\n" + "=" * 70)
    print(texte)
    print("=" * 70)


def afficher_metriques(nom, y_vrai, y_pred, y_proba=None):
    """Affiche les métriques principales d'un classifieur binaire."""
    print(f"\n--- {nom} ---")
    print(f"Accuracy  : {accuracy_score(y_vrai, y_pred):.3f}  (part de prédictions correctes)")
    print(f"Précision : {precision_score(y_vrai, y_pred):.3f}  (parmi les 'malin' prédits, combien le sont vraiment)")
    print(f"Rappel    : {recall_score(y_vrai, y_pred):.3f}  (parmi les vrais 'malin', combien sont détectés)")
    print(f"F1-score  : {f1_score(y_vrai, y_pred):.3f}  (moyenne harmonique précision / rappel)")
    if y_proba is not None:
        print(f"ROC AUC   : {roc_auc_score(y_vrai, y_proba):.3f}  (capacité à classer malin avant bénin, 1 = parfait)")
    vn, fp, fn, vp = confusion_matrix(y_vrai, y_pred).ravel()
    print(f"Matrice de confusion : VN={vn}  FP={fp}  FN={fn}  VP={vp}")
    print(f"  -> {fn} tumeur(s) maligne(s) ratée(s) (faux négatifs, l'erreur la plus grave ici)")
    print(f"  -> {fp} fausse(s) alerte(s) (faux positifs)")


titre("1. Préparation des données")
data = load_breast_cancer()
X = data.data
# Dans scikit-learn 0 = malin, 1 = bénin. On inverse pour que la classe
# "positive" (1) soit la tumeur maligne, celle qu'on veut détecter.
y = 1 - data.target

print(f"{X.shape[0]} patientes, {X.shape[1]} variables")
print(f"Malin : {y.sum()}  |  Bénin : {(y == 0).sum()}")
print("Exemple d'échelles très différentes entre variables :")
print(f"  'mean area' va de {X[:, 3].min():.0f} à {X[:, 3].max():.0f}")
print(f"  'mean smoothness' va de {X[:, 4].min():.3f} à {X[:, 4].max():.3f}")

# Jeu de test mis de côté : il ne sert QU'À l'évaluation finale.
# stratify=y garde la même proportion malin / bénin dans les deux jeux.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
print(f"Entraînement : {len(X_train)}  |  Test : {len(X_test)}")


titre("2. KNN (K=5) SANS normalisation")
# "Entraîner" un KNN = simplement mémoriser X_train et y_train.
knn_brut = KNeighborsClassifier(n_neighbors=5)
knn_brut.fit(X_train, y_train)
y_pred_brut = knn_brut.predict(X_test)
afficher_metriques("KNN sans normalisation", y_test, y_pred_brut,
                   knn_brut.predict_proba(X_test)[:, 1])
print("\nProblème : 'mean area' (des centaines) écrase 'mean smoothness' (des")
print("centièmes) dans le calcul de la distance. Le modèle ignore donc une partie de l'info.")


titre("3. KNN (K=5) AVEC normalisation")
# Le Pipeline garantit que le StandardScaler apprend moyenne / écart-type
# uniquement sur les données d'entraînement (pas de fuite d'information du test).
knn_norm = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
knn_norm.fit(X_train, y_train)
y_pred_norm = knn_norm.predict(X_test)
y_proba_norm = knn_norm.predict_proba(X_test)[:, 1]
afficher_metriques("KNN normalisé", y_test, y_pred_norm, y_proba_norm)

titre("4. Rapport détaillé par classe")
print(classification_report(y_test, y_pred_norm, target_names=["bénin", "malin"], digits=3))


titre("5. Erreur d'entraînement vs erreur de validation selon K")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
valeurs_k = range(1, 41)
err_train, err_val = [], []
for k in valeurs_k:
    modele = Pipeline([("scaler", StandardScaler()),
                       ("knn", KNeighborsClassifier(n_neighbors=k))])
    modele.fit(X_train, y_train)
    err_train.append(1 - modele.score(X_train, y_train))
    # Erreur de validation = moyenne sur 5 plis de validation croisée
    err_val.append(1 - cross_val_score(modele, X_train, y_train, cv=cv).mean())

for k in [1, 3, 5, 9, 15, 25, 40]:
    print(f"K={k:>2}  erreur entraînement = {err_train[k-1]:.3f}   erreur validation = {err_val[k-1]:.3f}")
k_min = int(np.argmin(err_val)) + 1
print(f"\nK=1 : erreur d'entraînement nulle (chaque point est son propre voisin) mais")
print("erreur de validation plus haute -> SURAPPRENTISSAGE (variance forte).")
print("K très grand : les deux erreurs remontent -> SOUS-APPRENTISSAGE (biais fort).")
print(f"Erreur de validation minimale pour K = {k_min}")

plt.figure(figsize=(8, 4.5))
plt.plot(valeurs_k, err_train, label="Erreur d'entraînement", color="#2a6fdb")
plt.plot(valeurs_k, err_val, label="Erreur de validation (CV 5 plis)", color="#e07a1f")
plt.axvline(k_min, color="grey", linestyle="--", linewidth=1, label=f"Meilleur K = {k_min}")
plt.xlabel("K (nombre de voisins)")
plt.ylabel("Taux d'erreur")
plt.title("Compromis biais / variance du KNN")
plt.legend()
plt.tight_layout()
plt.savefig("erreur_selon_k.png", dpi=150)
plt.close()


titre("6. Réglage de K, de la pondération et de la distance (GridSearchCV)")
grille = {
    "knn__n_neighbors": list(range(1, 31)),
    "knn__weights": ["uniform", "distance"],   # distance = les voisins proches comptent plus
    "knn__p": [1, 2],                          # 1 = Manhattan, 2 = euclidienne
}
recherche = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]),
    grille, cv=cv, scoring="f1", n_jobs=-1,
)
recherche.fit(X_train, y_train)
print(f"Meilleurs paramètres : {recherche.best_params_}")
print(f"F1 moyen en validation croisée : {recherche.best_score_:.3f}")

meilleur = recherche.best_estimator_
y_pred_best = meilleur.predict(X_test)
y_proba_best = meilleur.predict_proba(X_test)[:, 1]
afficher_metriques("KNN optimisé (évaluation finale sur le jeu de test)", y_test, y_pred_best, y_proba_best)


titre("7. Réduire les faux négatifs en abaissant le seuil")
# predict() classe "malin" si plus de 50 % des voisins sont malins.
# En médecine, rater un cancer coûte plus cher qu'une fausse alerte :
# on peut donc déclarer "malin" dès 30 % de voisins malins.
# Le seuil est choisi ici sur des prédictions de validation croisée,
# jamais sur le jeu de test.
# Remarque : avec K voisins, la "probabilité" ne prend que K+1 valeurs
# (avec K=3 : 0, 1/3, 2/3, 1). Des seuils voisins donnent donc le même résultat.
from sklearn.model_selection import cross_val_predict
proba_cv = cross_val_predict(meilleur, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
print("Seuil  Rappel(CV)  Précision(CV)")
for s in [0.5, 0.4, 0.3, 0.2]:
    p = (proba_cv >= s).astype(int)
    print(f" {s:.1f}     {recall_score(y_train, p):.3f}       {precision_score(y_train, p):.3f}")
seuil = 0.3
y_pred_seuil = (y_proba_best >= seuil).astype(int)
afficher_metriques(f"KNN optimisé avec seuil = {seuil} (jeu de test)", y_test, y_pred_seuil, y_proba_best)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best, display_labels=["bénin", "malin"], cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title("Matrice de confusion (seuil 0,5)")
axes[0].set_xlabel("Classe prédite")
axes[0].set_ylabel("Classe réelle")
RocCurveDisplay.from_predictions(y_test, y_proba_best, ax=axes[1], name="KNN optimisé")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey", label="Hasard")
axes[1].set_title("Courbe ROC")
axes[1].set_xlabel("Taux de faux positifs")
axes[1].set_ylabel("Taux de vrais positifs (rappel)")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.savefig("confusion_et_roc.png", dpi=150)
plt.close()


titre("8. Bonus : KNN en régression (progression du diabète à 1 an)")
Xr, yr = load_diabetes(return_X_y=True)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=SEED)
reg = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]),
    {"knn__n_neighbors": list(range(1, 41)), "knn__weights": ["uniform", "distance"]},
    cv=5, scoring="neg_root_mean_squared_error",
)
reg.fit(Xr_tr, yr_tr)
yr_pred = reg.predict(Xr_te)
base = np.full_like(yr_te, yr_tr.mean(), dtype=float)  # modèle naïf : toujours la moyenne
print(f"Meilleurs paramètres : {reg.best_params_}")
print(f"MAE  : {mean_absolute_error(yr_te, yr_pred):.1f}   (erreur absolue moyenne)")
print(f"RMSE : {np.sqrt(mean_squared_error(yr_te, yr_pred)):.1f}   (pénalise plus les grosses erreurs)")
print(f"R²   : {r2_score(yr_te, yr_pred):.3f}  (part de variance expliquée)")
print(f"Référence naïve (toujours prédire la moyenne) : RMSE = {np.sqrt(mean_squared_error(yr_te, base)):.1f}, R² = {r2_score(yr_te, base):.3f}")

print("\nGraphiques enregistrés : erreur_selon_k.png, confusion_et_roc.png")


1. Préparation des données
569 patientes, 30 variables
Malin : 212  |  Bénin : 357
Exemple d'échelles très différentes entre variables :
  'mean area' va de 144 à 2501
  'mean smoothness' va de 0.053 à 0.163
Entraînement : 426  |  Test : 143

2. KNN (K=5) SANS normalisation

--- KNN sans normalisation ---
Accuracy  : 0.923  (part de prédictions correctes)
Précision : 0.977  (parmi les 'malin' prédits, combien le sont vraiment)
Rappel    : 0.811  (parmi les vrais 'malin', combien sont détectés)
F1-score  : 0.887  (moyenne harmonique précision / rappel)
ROC AUC   : 0.956  (capacité à classer malin avant bénin, 1 = parfait)
Matrice de confusion : VN=89  FP=1  FN=10  VP=43
  -> 10 tumeur(s) maligne(s) ratée(s) (faux négatifs, l'erreur la plus grave ici)
  -> 1 fausse(s) alerte(s) (faux positifs)

Problème : 'mean area' (des centaines) écrase 'mean smoothness' (des
centièmes) dans le calcul de la distance. Le modèle ignore donc une partie de l'info.

3. KNN (K=5) AVEC normalisation

--- KN

In [2]:
!pip install -q gradio

In [3]:
import gradio as gr
import pandas as pd

# Nous allons utiliser le meilleur modèle issu de notre GridSearchCV
# (ou le modèle normalisé si la recherche n'a pas été entièrement sauvegardée dans une variable globale)
try:
    model_to_use = meilleur
    seuil_optimal = 0.3
    print("Utilisation du modèle KNN optimisé par GridSearch avec seuil de 0.3.")
except NameError:
    model_to_use = knn_norm
    seuil_optimal = 0.5
    print("Utilisation du modèle KNN normalisé par défaut.")

# Sélectionner quelques caractéristiques importantes pour simplifier l'interface utilisateur
# Le jeu de données complet contient 30 caractéristiques. Pour éviter une interface trop lourde,
# nous allons proposer de saisir les valeurs pour les 5 variables les plus importantes historiquement
# ou bien d'utiliser une moyenne pour les autres.

feature_names = list(data.feature_names)
mean_values = X_train.mean(axis=0)

# Indices des variables clés à afficher sur l'interface
key_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness"
]
key_indices = [feature_names.index(name) for name in key_features]

def predict_breast_cancer(*args):
    # Reconstruire le vecteur complet de 30 caractéristiques
    full_input = mean_values.copy()

    # Mettre à jour les variables clés avec les saisies de l'utilisateur
    for idx, val in zip(key_indices, args):
        full_input[idx] = val

    # Prédiction avec le modèle
    input_array = full_input.reshape(1, -1)
    proba_malin = model_to_use.predict_proba(input_array)[0, 1]

    # Classification selon le seuil
    prediction = "Maligne (Cancer)" if proba_malin >= seuil_optimal else "Bénigne (Sain)"

    # Formatage du résultat pour l'affichage Gradio
    resultat = {
        "Bénigne": 1.0 - proba_malin,
        "Maligne": proba_malin
    }

    message = f"**Diagnostic suggéré :** {prediction}\n\n*(Seuil de décision pour la classe Maligne réglé à {seuil_optimal:.1f} pour minimiser les faux négatifs)*"
    return message, resultat

# Création des champs de saisie dynamiques basés sur les plages réelles des données
inputs = []
for name in key_features:
    idx = feature_names.index(name)
    min_val = float(X[:, idx].min())
    max_val = float(X[:, idx].max())
    default_val = float(X_train[:, idx].mean())
    inputs.append(
        gr.Slider(
            minimum=round(min_val, 3),
            maximum=round(max_val, 3),
            value=round(default_val, 3),
            label=name.replace(" ", " ").title()
        )
    )

outputs = [
    gr.Markdown(label="Diagnostic"),
    gr.Label(label="Probabilités")
]

# Interface Gradio
demo = gr.Interface(
    fn=predict_breast_cancer,
    inputs=inputs,
    outputs=outputs,
    title="🩺 Détection du Cancer du Sein (KNN Classifier)",
    description="Ajustez les curseurs pour modifier les caractéristiques de la tumeur et voir la prédiction du modèle en temps réel.",
    live=True
)

# Lancement de l'application
demo.launch(quiet=True, share=True)

Utilisation du modèle KNN optimisé par GridSearch avec seuil de 0.3.
* Running on public URL: https://ae673e54f314442819.gradio.live
